# 11 · Adaptation after a distribution switch

**What this shows.** What happens to the criterion (PSE) when the stimulus distribution changes between
blocks — Uniform → Hard-A → Hard-B → Hard-A — in the pre-opto cohorts: the size of the shift, how fast it
converges within a block (the manuscript's Fig. 5C recipe), what carries over overnight, and the
per-session psychometric parameters. This is the behaviour the PPC manipulation is meant to perturb.

**What produced it.** A `switch_adaptation` run:

```bash
sc-reports switch-adaptation --cohort behaviour1-cohort
```

which writes, under `switch_adaptation/<cohort>/<run_id>/switches/`: `switches.csv` (one row per switch
and stat), `pre_post.csv` (PSE before and after each switch, with the normative PSE), `convergence.csv`
(PSE against trials since the switch, per method), `sessions.csv` (per-session stats in block order),
`overnight.csv` (end-of-day to next-morning PSE), `psychometrics.csv` + `psychometric_curves.csv`
(per-phase fits), and PDFs per animal plus `summary_switches.pdf`.

**Design caveat.** Blocks were long in these cohorts; the opto cohort used a daily A/B alternation
instead, which is why notebook 21 shows no measurable adaptation there. The blocked design is what the
post-shift opto experiment needs.

In [ ]:
from nb_setup import *
banner()
run = open_run('switch_adaptation', SWITCH_COHORT)
T = load_switches(run)
{k: v.shape for k, v in T.items()}

## 1 · One animal: where the switches are, and the pipeline

`behaviour/adaptation.py` finds switches from the session table (`animal_switches`), then
`compute_switches` fits the PSE before and after each one and the convergence curve within the new block.

In [ ]:
from sound_categorisation.behaviour.adaptation import animal_switches

exp = load_experiment()
aid = sorted(T['switches'].animal.unique())[0]
animal = exp.animals[aid]
sessions, switches = animal_switches(animal, stage=None if SYNTHETIC else 'Full_Task_Cont', min_block_trials=300)
print(f'{aid}: {len(sessions)} sessions, {len(switches)} switches')
pd.DataFrame(switches).head(10) if switches else 'no switches'

## 2 · Pre/post PSE per switch, every animal

`pre_post.csv`: the PSE fitted on the last trials before the switch and the first after it, against the
normative PSE for the new distribution (the criterion an ideal observer of that distribution would adopt).

In [ ]:
pp = T['pre_post']
display(pp[['animal', 'switch_idx', 'transition', 'pse_pre', 'pse_post', 'normative_pse', 'delta']].round(3))

fig, ax = plt.subplots(figsize=(6, 3.5))
for tr, sub in pp.groupby('transition'):
    ax.plot([0, 1], [sub['pse_pre'], sub['pse_post']], color='0.6', lw=0.8, alpha=0.6)
    ax.scatter([1] * len(sub), sub['pse_post'], label=tr, s=30, zorder=3)
ax.set(xticks=[0, 1], xticklabels=['pre', 'post'], ylabel='PSE', title='PSE before and after each switch')
ax.axhline(0, color='0.8', lw=0.6); ax.legend(fontsize=8)

## 2b · Who adapts, who doesn't, who goes the wrong way

Not every animal shifts its criterion toward the new distribution. For each switch, `sign(delta)` is
compared with the direction the normative PSE moved (`normative_pse − pse_pre`): **toward** (shifted the
right way), **none** (|delta| below a tolerance), **reverse** (shifted the wrong way). Per animal, the
tally over its switches; the odd ones are listed. These animals are the ones to look at individually in
the per-animal PDFs (`switches/pdf/<animal>_switches.pdf`) before any group claim is read.


In [ ]:
TOL = 0.05                                             # |delta| below this counts as no shift
pp = T['pre_post'].copy()
expected = np.sign(pp['normative_pse'] - pp['pse_pre'])
pp['direction'] = np.where(pp['delta'].abs() < TOL, 'none',
                   np.where(np.sign(pp['delta']) == expected, 'toward', 'reverse'))
tally = pp.pivot_table(index='animal', columns='direction', values='switch_idx', aggfunc='count', fill_value=0)
for col in ('toward', 'none', 'reverse'):
    if col not in tally:
        tally[col] = 0
tally['n_switches'] = tally[['toward', 'none', 'reverse']].sum(axis=1)
tally['fraction_toward'] = (tally['toward'] / tally['n_switches']).round(2)
display(tally[['n_switches', 'toward', 'none', 'reverse', 'fraction_toward']].sort_values('fraction_toward'))

odd = tally[tally['fraction_toward'] < 0.5]
print('adapt the expected way on fewer than half their switches:', list(odd.index) or 'nobody')
pp[pp['animal'].isin(odd.index)][['animal', 'switch_idx', 'transition', 'pse_pre', 'pse_post', 'normative_pse', 'delta', 'direction']].round(3)


## 3 · Convergence within the new block (Fig. 5C)

`convergence.csv`: PSE as a function of trials since the switch, per switch and method, and
`convergence` = how far the shift has gone toward the normative PSE (0 = still at the pre-switch value,
1 = at the normative value). Mean over animals per transition, with the individual switches faint.

In [ ]:
cv = T['convergence']
methods = sorted(cv['method'].unique())
method = methods[0]
sub = cv[cv['method'] == method]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for tr, s in sub.groupby('transition'):
    for (a, k), ss in s.groupby(['animal', 'switch_idx']):
        axes[0].plot(ss['trial'], ss['convergence_clipped'], color='0.75', lw=0.6)
    m = s.groupby('trial')['convergence_clipped'].mean()
    axes[0].plot(m.index, m.values, lw=2, label=tr)
axes[0].axhline(1, color='0.5', lw=0.6, ls='--'); axes[0].axhline(0, color='0.5', lw=0.6)
axes[0].set(xlabel='trials since switch', ylabel='convergence (0 = pre, 1 = normative)', title=f'method = {method}')
axes[0].legend(fontsize=8)
# per-switch trials-to-90% as a distribution
sw = T['switches']
t90 = sw[sw['stat'].str.contains('trials_to_90')]
if len(t90):
    axes[1].hist(t90['value'].dropna(), bins=12, color='0.6')
    axes[1].set(xlabel='trials to 90 % of the shift', ylabel='switches')
plt.tight_layout()

## 4 · Session by session inside a block, and overnight

`sessions.csv` gives every stat per session in block order (`order_in_block`); `overnight.csv` the PSE at
the end of a session against the start of the next — does the shift survive the night, or does the animal
drift back toward the previous distribution (`toward_pre`)?

In [ ]:
se = T['sessions']
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for (a, k), s in se.groupby(['animal', 'switch_idx']):
    s = s.sort_values('order_in_block')
    axes[0].plot(s['order_in_block'], s['pse_fixed'] - s['pse_fixed'].iloc[0], color='0.6', lw=0.8, alpha=0.6)
axes[0].axhline(0, color='0.5', lw=0.6)
axes[0].set(xlabel='session in block', ylabel='PSE − first session of block', title='within-block drift')
ov = T['overnight']
axes[1].scatter(ov['pse_end'], ov['pse_start_next'], s=20)
lim = np.nanmax(np.abs(ov[['pse_end', 'pse_start_next']].to_numpy())) * 1.1 or 1
axes[1].plot([-lim, lim], [-lim, lim], color='0.6', lw=0.8)
axes[1].set(xlabel='PSE at end of session', ylabel='PSE at start of next', title=f'overnight: {int(ov["toward_pre"].sum())}/{len(ov)} moved back toward pre')
plt.tight_layout()

## 5 · Per-phase psychometrics across animals

The fitted curves per phase (block) for every animal, with the fit parameters. This is the group picture
the opto post-shift experiment compares against.

In [ ]:
ps, pc = T['psychometrics'], T['psychometric_curves']
dists = [d for d in pc['distribution'].unique() if isinstance(d, str)]
fig, axes = plt.subplots(1, len(dists), figsize=(3.6 * len(dists), 3.2), sharey=True, squeeze=False)
for ax, d in zip(axes[0], dists):
    for (a, ph), s in pc[pc['distribution'] == d].groupby(['animal', 'phase']):
        ax.plot(s['x'], s['y'], color='0.6', lw=0.7, alpha=0.6)
    m = pc[pc['distribution'] == d].groupby('x')['y'].mean()
    ax.plot(m.index, m.values, lw=2, color=PALETTE[0])
    ax.axhline(0.5, color='0.8', lw=0.5); ax.axvline(0, color='0.8', lw=0.5)
    ax.set(title=d, xlabel='stimulus')
axes[0, 0].set_ylabel('P(choose B)')
plt.tight_layout()
ps.groupby('distribution')[['mu', 'sigma', 'accuracy', 'hard_accuracy']].agg(['median', 'count']).round(3)